# Original ApET: fixed GQA subsets

First stop the full-run process and download its partial ZIP. Enable Internet and T4 ×2. Default: development, 800 questions on 100 images, four configurations = 3,200 predictions. Evaluation has 800 questions on separate images; keep it for final comparisons. Native code, dependencies and scoring stay unchanged. See `research/GQA_SUBSETS.md`.

## 1. Choose a set and a new run directory


In [ ]:
import json
import subprocess
from datetime import datetime, timezone
from pathlib import Path

RUN_SUBSET_TEST = False  # Set True when ready to run these cells.
SUBSET = "development"  # "evaluation" is reserved for final comparisons.
CONFIGURATIONS = ["baseline", "input-only", "decoder-only", "combined"]
REFERENCE_REPORT = None  # Later: path to completed baseline JSON; also choose e.g. CONFIGURATIONS=["combined"].
RESUME_RUN_DIR = None  # Optional extracted prior subset ZIP directory.
REPO_URL = "https://github.com/doguineyes/CSE590-ApET.git"
REPO_REF = "apet-analysis"
if not RUN_SUBSET_TEST:
    raise RuntimeError("Set RUN_SUBSET_TEST=True after stopping/downloading the old run.")
if SUBSET not in {"development", "evaluation"}:
    raise ValueError("Choose development or evaluation")

repo = Path("/kaggle/working/CSE590-ApET")
if RESUME_RUN_DIR:
    run_dir = Path(RESUME_RUN_DIR)
    saved = json.loads((run_dir / "original-gqa-subset.json").read_text())
    REPO_REF = saved["commit"]
    SUBSET = saved["dataset"]["study"]["role"]
    CONFIGURATIONS = saved["protocol"]["configuration_order"]
    REFERENCE_REPORT = None  # The runner restores any saved baseline from the ZIP snapshot.
else:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    run_dir = Path(f"/kaggle/working/original-gqa-{SUBSET}-{stamp}")
    run_dir.mkdir()
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
subprocess.run(["git", "fetch", "origin"], cwd=repo, check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=repo, check=True)
if not RESUME_RUN_DIR and REPO_REF == "apet-analysis":
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True)
original_python = repo / "environments/original/.venv/bin/python"
study_dir = Path("/kaggle/working/gqa-fixed-subsets-v1-s590")
data_manifest = (run_dir / "original-gqa-subset-data/manifest.json") if RESUME_RUN_DIR else study_dir / SUBSET / "manifest.json"
report_path = run_dir / "original-gqa-subset.json"
checkpoint_args = []  # Optional attached original-format LLaVA/CLIP paths.
print("Set:", SUBSET, "Configurations:", CONFIGURATIONS, "Run directory:", run_dir)


## 2. Reuse the environment and prepare frozen sets

Only the existing two pinned source files (~68 MB) are needed. Preparation uses metadata and seed 590 without reading predictions. Each development/evaluation set has 100 images × 8 questions; an 80-question diagnostic set is nested in development. The inspected pilot images are excluded from evaluation.


In [ ]:
import shutil

subprocess.run([
    "bash", "bootstrap/setup_original_linux.sh", "--require-cuda", "--expected-gpus", "2",
    "--output", str(run_dir / "original-runtime.json"),
], cwd=repo, check=True)
subprocess.run([str(original_python), "-I", "research/tests/test_gqa_subsets.py"], cwd=repo, check=True)
if not data_manifest.is_file():
    if RESUME_RUN_DIR:
        raise FileNotFoundError(f"Restored subset data missing: {data_manifest}")
    subprocess.run([
        str(original_python), "-I", "scripts/prepare_gqa_subsets.py",
        "--output-dir", str(study_dir), "--seed", "590", "--no-zip",
    ], cwd=repo, check=True)
data = json.loads(data_manifest.read_text())
print("Study ID:", data["study"]["id"])
print("Images:", len(data["images"]), "Questions:", len(data["questions"]))
print("Types:", data["selection"]["type_counts"]["semantic"])
print("New predictions:", len(data["questions"]) * len(CONFIGURATIONS))


## 3. Run or resume the selected configurations

Initial development run: 3,200 predictions, approximately 30 minutes based on the pilot (actual runtime varies). Loads the model once across both T4s. JSONL flushes per prediction; summaries and ETA refresh every 50 questions. Rerun this cell to resume. Saved baselines are used for accuracy/APR; their timings belong to their original session.


In [ ]:
import signal

resume_args = ["--resume"] if report_path.exists() else []
reference_args = ["--reference-report", str(REFERENCE_REPORT)] if REFERENCE_REPORT else []
command = [
    str(original_python), "-I", "scripts/eval_original_gqa_subset.py",
    "--data-manifest", str(data_manifest), "--configurations", *CONFIGURATIONS,
    "--max-new-tokens", "16", "--download-budget-gib", "20", "--report-every", "50",
    *checkpoint_args, *reference_args, *resume_args, "--output", str(report_path),
]
process = subprocess.Popen(command, cwd=repo)
try:
    returncode = process.wait()
except KeyboardInterrupt:
    if process.poll() is None:
        process.send_signal(signal.SIGINT)
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    print("Stopped. Run Cell 4 to preserve partial evidence.")
    raise
if returncode:
    raise subprocess.CalledProcessError(returncode, command)


## 4. Inspect and download evidence

Includes raw predictions, selected images, full question metadata, the frozen split manifest, source snapshots and runtime records. Cached-reference runs also copy baseline records and source snapshots. Excludes packages and weights. Download before stopping the session. Keep the initial baseline ZIP for future comparisons.


In [ ]:
import zipfile
from IPython.display import FileLink, display

if "process" in globals() and process.poll() is None:
    raise RuntimeError("Stop the evaluator before creating its ZIP.")
if report_path.is_file():
    result = json.loads(report_path.read_text())
    print("Status:", result["status"], "Progress:", result.get("progress"))
    print("Baseline origin:", result.get("baseline_origin"))
    if result.get("error"):
        print("Error:", result["error"])
    for name, metrics in result.get("summaries", {}).items():
        print(name, metrics["correct"], "/", metrics["evaluated"], "accuracy:", metrics["accuracy"],
              "APR:", result.get("comparisons", {}).get(name, {}).get("apr"))
bundle = run_dir.with_suffix(".zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(run_dir.rglob("*")):
        if path.is_file() and path.suffix != ".lock":
            archive.write(path, str(path.relative_to(run_dir)))
print("Download before stopping the session:", bundle)
display(FileLink(str(bundle)))
